In [4]:
# === Small BERT example (Colab, 1 cell) ===
# If you just installed packages, do Runtime → Restart runtime before running.

import torch, transformers, datasets, inspect
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__, "| Datasets:", datasets.__version__)

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import os

# Disable Weights & Biases auto-logging so it won't prompt for an API key
os.environ["WANDB_DISABLED"] = "true"

# 1) Load a tiny dataset split to train fast: tweet_eval/sentiment (3 labels: 0=neg,1=neu,2=pos)
ds = load_dataset("tweet_eval", "sentiment")
rng = 42
# Use small subsets to keep runtime short on Colab
train_small = ds["train"].shuffle(seed=rng).select(range(400))
valid_small = ds["validation"].shuffle(seed=rng).select(range(200))
test_small  = ds["test"].shuffle(seed=rng).select(range(200))

# 2) Tokenizer and model
model_name = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(model_name)

# Simple preprocessing: pad/truncate to a fixed length for batching
def preprocess(batch):
    return tok(batch["text"], truncation=True, padding="max_length", max_length=128)

# Convert HF Datasets to PyTorch tensors with the expected column names
def to_torch(split):
    d = split.map(preprocess, batched=True)
    d = d.rename_column("label","labels")  # HF Trainer expects "labels"
    d.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
    return d

train_tok = to_torch(train_small)
valid_tok = to_torch(valid_small)
test_tok  = to_torch(test_small)

# Create a sequence classification head on top of BERT. num_labels=3 for sentiment
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Metrics for evaluation (macro-F1 is robust to class imbalance)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

# 3) TrainingArguments with compatibility guard:
# Some older transformers versions lack "evaluation_strategy".
# We inspect the signature and only pass supported args.
sig_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())

kwargs = dict(
    output_dir="out-bert-small",       # where checkpoints/logs are saved
    learning_rate=2e-5,                # common LR for BERT fine-tuning
    per_device_train_batch_size=16,    # small batch keeps memory low
    per_device_eval_batch_size=32,     # larger eval batch is fine if memory allows
    num_train_epochs=2,                # quick demo (increase for better scores)
    weight_decay=0.01,                 # mild regularization
)

# Enable mixed precision if supported and GPU is available (faster/less memory)
if "fp16" in sig_params:
    kwargs["fp16"] = torch.cuda.is_available()
elif "bf16" in sig_params:
    kwargs["bf16"] = torch.cuda.is_available()

# Choose evaluation API depending on transformers version
if "evaluation_strategy" in sig_params:
    kwargs.update(
        evaluation_strategy="epoch",   # run eval at the end of each epoch
        save_total_limit=1,            # keep only the best/latest checkpoint(s)
        load_best_model_at_end=True,   # restore best checkpoint by metric below
        metric_for_best_model="macro_f1",
        report_to="none",              # avoid wandb/tensorboard logging
        logging_steps=50,              # periodic console logging
    )
elif "evaluate_during_training" in sig_params:
    kwargs.update(
        evaluate_during_training=True, # legacy flag in older versions
        eval_steps=200,
        logging_steps=50,
    )

args = TrainingArguments(**kwargs)

# HF Trainer handles training loop, evaluation, and device placement
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    # If neither eval API exists, we can still evaluate after training explicitly
    eval_dataset=valid_tok if ("evaluation_strategy" in sig_params or "evaluate_during_training" in sig_params) else None,
    compute_metrics=compute_metrics
)

# 4) Train and evaluate
trainer.train()
print("VALID:", trainer.evaluate(valid_tok))
print("TEST :", trainer.evaluate(test_tok))

# 5) Quick inference helper
id2label = {0:"negative",1:"neutral",2:"positive"}

def predict(texts):
    # Tokenize and move tensors to the same device as the model
    batch = tok(texts, truncation=True, padding=True, max_length=128, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        logits = trainer.model(**batch).logits
    # Map predicted class ids to human-readable labels
    return [id2label[i] for i in logits.argmax(dim=-1).cpu().tolist()]

print(predict(["I loved this!", "It's okay.", "Worst thing ever."]))


Torch: 2.8.0+cu126 CUDA: False
Transformers: 4.56.1 | Datasets: 4.0.0


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


VALID: {'eval_loss': 0.9587873816490173, 'eval_accuracy': 0.525, 'eval_macro_f1': 0.3632748538011696, 'eval_runtime': 53.4179, 'eval_samples_per_second': 3.744, 'eval_steps_per_second': 0.131, 'epoch': 2.0}


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TEST : {'eval_loss': 1.1213668584823608, 'eval_accuracy': 0.455, 'eval_macro_f1': 0.31270247229326514, 'eval_runtime': 53.0488, 'eval_samples_per_second': 3.77, 'eval_steps_per_second': 0.132, 'epoch': 2.0}
['positive', 'neutral', 'neutral']
